# Step 3A - Dev Runtime Baselines on `dev_tune_200`

Step 2A established G0 candidate support. Step 2B established base/self-consistency and the `SelfLoc_base` denominator.

This notebook runs the first real dev-only runtime baseline comparison. It uses only:

```text
runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl
```

It does not touch `analysis_500`, `ablation_500`, `final_test_500`, or `final_test_full`.

Goal:

```text
1. Reuse the existing base seed0 run as the base method.
2. Run target_logit_bias in its own isolated run.
3. Run target-support/text-memory baselines separately: target_candidate_insert, prompt_memory.
4. Run non-MC bridge controls: myopic_score, no_rollout_bridge.
5. Run mc_bridge with the frozen initial config.
6. Run raw_bridge_gated with a simple subject gate as an implementation/dev baseline.
7. Build edit-balanced aggregate tables, target-length tables, candidate-coverage tables, and paired bootstrap deltas.
```

This is still dev tuning territory. Do not use any numbers here as final-test evidence.


In [ ]:
%%bash
set -euo pipefail

# Colab usually already provides CUDA-compatible torch, so do not reinstall
# unpinned torch unless the environment is missing it.
pip install -q   "transformers==4.46.3"   "datasets>=4.0.0"   "accelerate>=1.11.0"   "sentencepiece>=0.2.0"   "packaging"   "bitsandbytes>=0.43.0"


In [ ]:
import subprocess
import torch, transformers, datasets, accelerate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

assert torch.cuda.is_available(), "This notebook needs a GPU runtime."
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Save Runtime Environment

In [ ]:
import json
from pathlib import Path
import torch, transformers, datasets, accelerate

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert project.exists(), f"Project directory missing after Drive mount: {project}"

env = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
}
out = project / "runs/counterfact_direction1_v1/runtime_environment_step3a.json"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)

print("Wrote:", out)
print(json.dumps(env, indent=2))


## Preflight: Project and Prior Artifacts

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert PROJECT_DIR.exists(), f"Missing project dir: {PROJECT_DIR}"

required_files = [
    "llada_runtime_editor_eval.py",
    "llada_counterfact_protocol.py",
    "llada_experiment_reports.py",
    "paper_protocol.md",
    "implementation_spec.md",
    "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl",
    "runs/counterfact_direction1_v1/protocol/protocol_manifest.json",
    "runs/counterfact_direction1_v1/dev_tune_200_base_coverage/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_base_coverage/candidate_coverage.jsonl",
    "runs/counterfact_direction1_v1/dev_tune_200_base_seed0/summary.json",
    "runs/counterfact_direction1_v1/dev_tune_200_base_seed0/per_case_results.jsonl",
    "runs/counterfact_direction1_v1/dev_tune_200_base_selfconsistency_report/base_selfconsistency_summary.json",
]

for name in required_files:
    file_path = PROJECT_DIR / name
    assert file_path.exists(), f"Missing required file: {file_path}"

try:
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=PROJECT_DIR,
        text=True,
    ).strip()
    print("git commit:", commit)
except Exception as exc:
    print("git commit unavailable:", repr(exc))

with (PROJECT_DIR / "runs/counterfact_direction1_v1/protocol/protocol_manifest.json").open() as f:
    manifest = json.load(f)
assert manifest["protocol_version"] == "counterfact_direction1_v1"
assert manifest["artifacts"]["dev_tune_200"]["summary"]["count"] == 200

selfloc_path = PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_base_selfconsistency_report/base_selfconsistency_summary.json"
with selfloc_path.open() as f:
    selfloc_report = json.load(f)
assert selfloc_report["protocol_version"] == "counterfact_direction1_v1"
assert selfloc_report["split_role"] == "dev_tune_200"
assert selfloc_report["main_selfloc_key"] == "sample_self_agreement_paired_edit_balanced"
assert selfloc_report.get("balance_unit_field") == "edit_id"
selfloc_base = float(selfloc_report["selfloc_base_near_far_overall"])
assert 0.0 < selfloc_base <= 1.0

base_seed0_cfg_path = PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_base_seed0/run_config.json"
with base_seed0_cfg_path.open() as f:
    base_seed0_cfg = json.load(f)
assert base_seed0_cfg["methods"] == ["base"]
assert base_seed0_cfg["seed"] == 0
assert base_seed0_cfg["split_role"] == "dev_tune_200"

print("Python:", sys.version)
print("Project dir OK:", PROJECT_DIR)
print("Protocol OK:", manifest["protocol_version"])
print("SelfLoc_base:", selfloc_base)
print("Base seed0 OK:", base_seed0_cfg_path)


## Write Dev Threshold Snapshot

This snapshot records the Step 2B base budgets used for dev inspection. It is not an analysis/final lock file.


In [ ]:
import json
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
selfloc_path = project / "runs/counterfact_direction1_v1/dev_tune_200_base_selfconsistency_report/base_selfconsistency_summary.json"
with selfloc_path.open() as f:
    selfloc_report = json.load(f)

base_tfpr = selfloc_report["base_TFPR_by_bucket_mean"]
base_malformed = selfloc_report["base_malformed_rate_by_bucket_mean"]

def plus_budget(values, amount):
    return {
        key: (None if value is None else float(value) + amount)
        for key, value in values.items()
    }

snapshot = {
    "protocol_version": "counterfact_direction1_v1",
    "split_role": "dev_tune_200",
    "selfloc_base": selfloc_report["selfloc_base_near_far_overall"],
    "main_selfloc_key": selfloc_report["main_selfloc_key"],
    "balance_unit_field": selfloc_report.get("balance_unit_field"),
    "base_TFPR_by_bucket_mean": base_tfpr,
    "target_false_positive_rate_budget_by_bucket": plus_budget(base_tfpr, 0.03),
    "base_malformed_rate_by_bucket_mean": base_malformed,
    "malformed_span_rate_budget": 0.05,
    "gpu_minutes_per_edit_budget": 2.0,
    "note": "Dev-only threshold snapshot derived from Step 2B. Not a final-test lock file.",
}

out = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_thresholds.json"
with out.open("w", encoding="utf-8") as f:
    json.dump(snapshot, f, indent=2)

print("Wrote:", out)
print(json.dumps(snapshot, indent=2))


## Overwrite Guards

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
run_dirs = [
    project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_logit_bias",
    project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_support",
    project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_bridge_controls",
    project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_mc_bridge",
    project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_raw_bridge_gated",
    project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report",
]

for d in run_dirs:
    if d.exists():
        raise RuntimeError(
            f"Run directory already exists: {d}. "
            "Delete it manually or create a new run name. Do not overwrite silently."
        )

print("Overwrite guard passed.")


## Step 3A.1: Target Logit Bias Baseline

Runs only:

```text
target_logit_bias
```

This run is isolated so `--target_logit_bias 5.0` cannot contaminate support-only or prompt-memory baselines.


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_logit_bias

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_logit_bias   --methods target_logit_bias   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --target_logit_bias 5.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_logit_bias/stdout.log


## Step 3A.2: Target Support and Prompt Memory Baselines

Runs:

```text
target_candidate_insert
prompt_memory
```

This run explicitly sets `--target_logit_bias 0.0` so it is support/text-memory only.


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_support

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_support   --methods target_candidate_insert,prompt_memory   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_support/stdout.log


## Step 3A.3: Non-MC Bridge Controls

Runs:

```text
myopic_score
no_rollout_bridge
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_bridge_controls

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_bridge_controls   --methods myopic_score,no_rollout_bridge   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 0   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_bridge_controls/stdout.log


## Step 3A.4: MC Bridge Main Dev Run

Initial frozen dev config:

```text
steps=4
bridge_topk=4
mc_rollouts=2
guidance_scale=1.0
reward_mode=soft_overlap
reward_beta=6.0
```


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_mc_bridge

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_mc_bridge   --methods mc_bridge   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 2   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_mc_bridge/stdout.log


## Step 3A.5: Raw Bridge Gated Dev Run

This uses a simple subject gate as a dev baseline. Gate tuning happens later on `dev_tune_200`; this is not an analysis or final gate selection.


In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_raw_bridge_gated

python -u llada_runtime_editor_eval.py   --model_id GSAI-ML/LLaDA-8B-Base   --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl   --output_dir runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_raw_bridge_gated   --methods raw_bridge_gated   --eval_samples 4   --steps 4   --bridge_topk 4   --mc_rollouts 2   --guidance_scale 1.0   --reward_mode soft_overlap   --reward_beta 6.0   --gate_mode subject   --target_logit_bias 0.0   --seed 0   --use_4bit 1   --dtype float16   --skip_candidate_coverage 1   2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_raw_bridge_gated/stdout.log


## Validate Runtime Run Configs

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
method_specific = {
    "dev_tune_200_runtime_baselines_target_logit_bias": {
        "methods": ["target_logit_bias"],
        "target_logit_bias": 5.0,
        "mc_rollouts": 0,
        "bridge_topk": 4,
    },
    "dev_tune_200_runtime_baselines_target_support": {
        "methods": ["target_candidate_insert", "prompt_memory"],
        "target_logit_bias": 0.0,
        "mc_rollouts": 0,
        "bridge_topk": 4,
    },
    "dev_tune_200_runtime_baselines_bridge_controls": {
        "methods": ["myopic_score", "no_rollout_bridge"],
        "target_logit_bias": 0.0,
        "mc_rollouts": 0,
        "bridge_topk": 4,
        "guidance_scale": 1.0,
        "reward_mode": "soft_overlap",
        "reward_beta": 6.0,
    },
    "dev_tune_200_runtime_baselines_mc_bridge": {
        "methods": ["mc_bridge"],
        "target_logit_bias": 0.0,
        "mc_rollouts": 2,
        "bridge_topk": 4,
        "guidance_scale": 1.0,
        "reward_mode": "soft_overlap",
        "reward_beta": 6.0,
    },
    "dev_tune_200_runtime_baselines_raw_bridge_gated": {
        "methods": ["raw_bridge_gated"],
        "target_logit_bias": 0.0,
        "mc_rollouts": 2,
        "bridge_topk": 4,
        "guidance_scale": 1.0,
        "reward_mode": "soft_overlap",
        "reward_beta": 6.0,
        "gate_mode": "subject",
    },
}

expected_protocol = {
    "protocol_version": "counterfact_direction1_v1",
    "edit_access": "given_at_edit_time",
    "training_access": "none",
    "hyperparameter_access": "dev_tune_only",
    "split_role": "dev_tune_200",
    "seed": 0,
    "model_id": "GSAI-ML/LLaDA-8B-Base",
    "eval_samples": 4,
}


def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


for run_name, expected in method_specific.items():
    methods = expected["methods"]
    run_dir = project / "runs/counterfact_direction1_v1" / run_name
    cfg_path = run_dir / "run_config.json"
    summary_path = run_dir / "summary.json"
    rows_path = run_dir / "per_case_results.jsonl"
    assert cfg_path.exists(), f"Missing config: {cfg_path}"
    assert summary_path.exists(), f"Missing summary: {summary_path}"
    assert rows_path.exists(), f"Missing per-case rows: {rows_path}"

    with cfg_path.open() as f:
        cfg = json.load(f)

    for key, value in expected_protocol.items():
        assert cfg.get(key) == value, f"{cfg_path}: expected {key}={value!r}, got {cfg.get(key)!r}"
    assert cfg["methods"] == methods, f"{cfg_path}: expected methods {methods}, got {cfg['methods']}"
    assert cfg["coverage_only"] is False
    assert "analysis_500" not in json.dumps(cfg, sort_keys=True)
    assert "final_test_500" not in json.dumps(cfg, sort_keys=True)

    rollout = cfg.get("rollout_config", {})
    for key, value in expected.items():
        if key == "methods":
            continue
        assert rollout.get(key) == value, (
            f"{cfg_path}: expected rollout_config.{key}={value!r}, got {rollout.get(key)!r}"
        )

    rows = list(read_jsonl(rows_path))
    assert rows, f"No rows in {rows_path}"
    row_methods = {r.get("method") for r in rows}
    assert row_methods == set(methods), f"{run_name}: expected row methods {methods}, got {row_methods}"

    by_method_bucket = defaultdict(set)
    for row in rows:
        method = row.get("method")
        bucket = row.get("bucket")
        edit_id = str(row.get("edit_id") or row.get("case_id"))
        if method in methods and bucket in {"rewrite", "near_locality", "far_locality"}:
            by_method_bucket[(method, bucket)].add(edit_id)

    for method in methods:
        for bucket in ["rewrite", "near_locality", "far_locality"]:
            n = len(by_method_bucket[(method, bucket)])
            assert n == 200, f"{run_name}/{method}/{bucket}: expected 200 edits, got {n}"

    if "raw_bridge_gated" in methods:
        variants = {r.get("method_variant") for r in rows if r.get("method") == "raw_bridge_gated"}
        assert variants == {"raw_bridge_gated_subject"}, variants

    print("Config and completeness OK:", run_name)

print("All Step 3A runtime configs and row completeness checks OK.")


## Step 3A.6: Build Edit-Balanced Report

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report

SELFLOC_BASE=$(python - <<'PY'
import json
from pathlib import Path
p = Path("runs/counterfact_direction1_v1/dev_tune_200_base_selfconsistency_report/base_selfconsistency_summary.json")
with p.open() as f:
    payload = json.load(f)
print(payload["selfloc_base_near_far_overall"])
PY
)

python -u llada_experiment_reports.py \
  --input \
    runs/counterfact_direction1_v1/dev_tune_200_base_coverage/summary.json \
    runs/counterfact_direction1_v1/dev_tune_200_base_seed0/summary.json \
    runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_logit_bias/summary.json \
    runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_target_support/summary.json \
    runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_bridge_controls/summary.json \
    runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_mc_bridge/summary.json \
    runs/counterfact_direction1_v1/dev_tune_200_runtime_baselines_raw_bridge_gated/summary.json \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report \
  --selfloc_base "$SELFLOC_BASE" \
  --bootstrap_baseline base \
  --bootstrap_candidates target_logit_bias target_candidate_insert prompt_memory myopic_score no_rollout_bridge mc_bridge raw_bridge_gated_subject \
  --bootstrap_metric exact_rate \
  --bootstrap_samples 2000 \
  --seed 0 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report/stdout.log


## Inspect Report Tables

In [ ]:
import csv
import json
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
report_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_runtime_baseline_report"
expected_report_files = [
    "report_summary.json",
    "aggregate_method_bucket.csv",
    "aggregate_target_length.csv",
    "aggregate_relation.csv",
    "selection_scores.csv",
    "paired_bootstrap.csv",
    "candidate_coverage_by_length.csv",
    "candidate_coverage_by_relation.csv",
]

for name in expected_report_files:
    path = report_dir / name
    assert path.exists(), f"Missing report artifact: {path}"

summary_path = report_dir / "report_summary.json"
aggregate_path = report_dir / "aggregate_method_bucket.csv"
selection_path = report_dir / "selection_scores.csv"
bootstrap_path = report_dir / "paired_bootstrap.csv"

with summary_path.open() as f:
    report_summary = json.load(f)
assert report_summary["protocol_version"] == "counterfact_direction1_v1"

with aggregate_path.open(newline="") as f:
    aggregate = list(csv.DictReader(f))
with selection_path.open(newline="") as f:
    selection = list(csv.DictReader(f))
with bootstrap_path.open(newline="") as f:
    bootstrap = list(csv.DictReader(f))

methods = {
    "base",
    "target_logit_bias",
    "target_candidate_insert",
    "prompt_memory",
    "myopic_score",
    "no_rollout_bridge",
    "mc_bridge",
    "raw_bridge_gated_subject",
}
observed_methods = {row["method"] for row in aggregate if row.get("method")}
missing = methods - observed_methods
assert not missing, f"Missing methods in aggregate report: {missing}"

for row in aggregate:
    if row["bucket"] in {"rewrite", "near_locality", "far_locality"}:
        assert int(float(row["num_edits"])) == 200, row

print("Report files OK:", expected_report_files)
print("Aggregate rows:", len(aggregate))
print("Selection rows:", len(selection))
print("Bootstrap rows:", len(bootstrap))

print()
print("Key method/bucket metrics:")
for row in aggregate:
    if row["bucket"] in {"rewrite", "declarative_paraphrases", "near_locality", "far_locality"}:
        print(
            row["method"],
            row["bucket"],
            "num_edits=", row.get("num_edits"),
            "exact=", row.get("mean_exact_rate"),
            "f1=", row.get("mean_token_f1"),
            "tfpr=", row.get("mean_target_false_positive_rate"),
            "kl=", row.get("mean_sparse_guidance_kl"),
        )

print()
print("Selection scores:")
for row in selection:
    print(row)

print()
print("Paired bootstrap deltas vs base:")
for row in bootstrap:
    if row.get("mean_delta"):
        print(row)


## Expected Outcome

You should get:

```text
runtime configs OK for every non-base run
aggregate_method_bucket.csv
aggregate_target_length.csv
aggregate_relation.csv
selection_scores.csv
paired_bootstrap.csv
candidate_coverage_by_length.csv
candidate_coverage_by_relation.csv
```

Stop if any run config is missing protocol fields, if any report row for rewrite/near/far has `num_edits != 200`, or if `mc_bridge`/`raw_bridge_gated_subject` fail to produce rows.

If this passes, Step 3A has produced the first dev-only baseline comparison. The next step is to inspect whether MC bridge beats the simple baselines on rewrite/paraphrase and on the edit/locality/sparse-KL tradeoff before deciding whether to run schedule or gate sweeps.
